<div style="border-left:5px solid #67CBEE;padding:10px 16px;background:#F7F9FB;color:#19222D;margin:12px 0"><b>AMD Radeon · YOLO26 · Day 01</b><br><span style="color:#536273">算法 → 数据 → 训练 → 评测 → 证据。每一个结论都应由实际运行结果支撑。</span><br><span style="color:#536273">Algorithm → Data → Training → Evaluation → Evidence. Every conclusion must be backed by real execution results.</span></div>

# 03 / Lab C · ROCm GPU 训练、微调与 Loss

**建议时长：50–70 分钟**　｜　学习路径：50–70 minutes

> 请按顺序运行；运行结果必须来自自己的 GPU 环境，不可把示例文字当作实测数据。

---

# 03 / Lab C · ROCm GPU Training, Fine-tuning & Loss

**Suggested time: 预训练权重 → 前后向训练 → 模型工件** | Learning path: pretrained weights → forward/backward training → model artifacts

> Run the cells in order; every result must come from your own GPU environment — never present the sample text as measured data.

## 实验目标
1. 在真实 AMD ROCm GPU 上运行 YOLO26n 预训练模型微调。
2. 先用 COCO8（4 train + 4 val）做**训练链路冒烟**，再训练 Lab B 数据。
3. 保存运行条件、`best.pt`、`last.pt`、训练曲线与 SHA256。
4. 解释 Loss、Epoch、Batch、学习率、过拟合和泛化；不得把 Smoke 成绩说成工业精度。

**注意**：本实验需要 GPU 和时间。推荐课前缓存模型/数据。`amp=False` 仅作 ROCm AMP 异常排查，不应在没有问题时随意更改对照配置。

---

## Goal
1. Fine-tune the YOLO26n pretrained model on a real AMD ROCm GPU.
2. **Smoke-test the training pipeline** on COCO8 (4 train + 4 val) first, then train on the Lab B dataset.
3. Persist run conditions, `best.pt`, `last.pt`, training curves and SHA256 hashes.
4. Explain loss, epochs, batches, learning rate, overfitting and generalization; never dress smoke-run numbers up as industrial accuracy.

**Note**: this lab needs a GPU and time; pre-caching model/data before class is recommended. `amp=False` exists for ROCm AMP troubleshooting only — do not flip comparison configs without a diagnosed problem.

In [ ]:
from pathlib import Path
import os
import sys
import json

candidates = [Path.cwd(), Path.cwd().parent, Path('/workspace/YOLO26_ROCm_Day1_Labs')]
package = next((p for p in candidates if (p / 'course/lab_utils.py').exists()), None)
if package is None:
    raise FileNotFoundError('请先将 ZIP 完整解压，并从项目目录或 notebooks/ 目录启动 Jupyter。')
sys.path.insert(0, str(package))
from course.lab_utils import (root, ensure_dirs, load_config, save_config, write_json,
                              write_csv, system_info, require_rocm, sha256)

home = ensure_dirs()
cfg = load_config()
print('Project:', package, '\nPersistent output:', home)

In [ ]:
from course.lab_utils import get_metrics_summary
import torch
import time
from ultralytics import YOLO

device = require_rocm()
assert cfg['dataset_yaml'] and Path(cfg['dataset_yaml']).is_file(), \
    '先完成 02 Lab B，或设置 YOLO_USER_DATA_YAML'
model_initial = (str(home / 'artifacts/yolo26n.pt')
                 if (home / 'artifacts/yolo26n.pt').is_file() else cfg['model'])
print('Training:', model_initial, 'dataset:', cfg['dataset_yaml'])

In [ ]:
# 非正式数据集，验证 GPU 实际能跑 Forward/Backward，不用于发布精度成绩。
RUN_SMOKE = True
if RUN_SMOKE:
    smoke_model = YOLO(model_initial)
    smoke_model.train(data='coco8.yaml', epochs=1, imgsz=320, batch=4, device=device,
                      workers=0, amp=False, project=str(home / 'runs/lab_c'), name='smoke',
                      exist_ok=True, plots=False, seed=42)
    print('GPU smoke completed; COCO8 is not a benchmark.')

In [ ]:
# 正式教学训练。为了让学生在课上完成闭环，默认 12 Epoch；高质量结果需课后延长训练。
RUN_BASELINE = True
BASELINE_EPOCHS = 12
train_kwargs = dict(data=cfg['dataset_yaml'], epochs=BASELINE_EPOCHS,
                    imgsz=512, batch=8, device=device, workers=0, patience=30,
                    optimizer='SGD', seed=42, deterministic=False,
                    amp=False, plots=True, project=str(home / 'runs/lab_c'),
                    name='baseline', exist_ok=True)
if RUN_BASELINE:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    baseline = YOLO(model_initial)
    train_output = baseline.train(**train_kwargs)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    peak_gib = torch.cuda.max_memory_allocated() / 2**30
    run_dir = Path(baseline.trainer.save_dir).resolve()
    weights = run_dir / 'weights/best.pt'
    assert weights.exists(), f'best.pt missing: {weights}'
    record = {'run_dir': str(run_dir), 'weights': str(weights), 'sha256': sha256(weights),
              'elapsed_seconds': round(elapsed, 2), 'peak_allocated_GiB': round(peak_gib, 3),
              'args': train_kwargs, 'system': system_info(),
              'warning': 'Peak memory is PyTorch allocated memory, not total device VRAM.'}
    write_json(home / 'reports/lab_c_baseline.json', record)
    save_config({'baseline_run': str(run_dir), 'baseline_weights': str(weights),
                 'baseline_train_settings': train_kwargs})
    print(record)
else:
    print('未执行训练。后续实验需要一个经过验证的 best.pt，请不要伪造训练结果。')

In [ ]:
# 阅读训练工件：不同版本可能输出不同的列名，请以实际 CSV 为准。
from PIL import Image
import pandas as pd

cfg = load_config()
run_dir = Path(cfg['baseline_run'])
csv_path = run_dir / 'results.csv'
if csv_path.exists():
    history = pd.read_csv(csv_path)
    display(history.tail(5))
else:
    print('Missing results.csv:', csv_path)
if (run_dir / 'results.png').exists():
    display(Image.open(run_dir / 'results.png').resize((1120, 530)))

In [ ]:
# 不混淆验证集与最终测试集：首次使用 split='val'
cfg = load_config()
best = YOLO(cfg['baseline_weights'])
metrics = best.val(data=cfg['dataset_yaml'], split='val', device=device,
                   imgsz=640, batch=8, workers=0, plots=True, verbose=False)
summary = get_metrics_summary(metrics)
summary['split'] = 'val'
summary['model_sha256'] = sha256(Path(cfg['baseline_weights']))
write_json(home / 'reports/lab_c_baseline_val.json', summary)
print('Validation only (NOT test):', summary)

### 为什么基线固定训练用 `imgsz=512`，评测用 `imgsz=640`？
训练输入尺寸属于训练方案，下一步 Lab E 会只改变训练尺寸；为了公平比较两组模型，Lab E 会用同一个 `imgsz=640` 做评测。这是一种课堂控制变量方案，不是所有工业评测任务必须如此。

### 思考题 / 面试必答
1. `best.pt` 与 `last.pt` 有什么区别？
2. 为什么训练 Loss 下降不能证明模型在新场景一定更好？
3. `torch.version.hip` 与 `torch.cuda.is_available()` 分别证明什么？
4. 如果 AMP 检查/训练有 NaN，你会怎样排查并记录配置变动？

**交付**：`runs/lab_c/baseline/`、`reports/lab_c_baseline.json`、`lab_c_baseline_val.json`。

---

### Why does the baseline train at `imgsz=512` but evaluate at `imgsz=640`?
Training input size is part of the training recipe — Lab E will vary exactly this knob. To compare both arms fairly, Lab E evaluates every model with the same `imgsz=640`. This is a classroom controlled-variable design, not a universal industrial rule.

### Think / Interview Questions
1. What is the difference between `best.pt` and `last.pt`?
2. Why doesn't a falling training loss prove the model is better on new scenes?
3. What do `torch.version.hip` and `torch.cuda.is_available()` each prove?
4. If the AMP check or training produces NaN, how do you troubleshoot and record the config change?

**Deliverables**: `runs/lab_c/baseline/`, `reports/lab_c_baseline.json`, `lab_c_baseline_val.json`.